# HTB Academy Module 302 - Skills Assessment Study Notebook

This is an annotated learning copy of the HTB student template. It trains the required four-class one-vs-rest logistic-regression model and demonstrates a label-only poisoning strategy. HTB already showed this assessment as completed when these notes were prepared; the accepted answer is not stored here.

## Before running

1. Download the assessment ZIP from the HTB question and extract assessment_dataset.npz into this notebook's folder.
2. Run the baseline, poisoning, evaluation, and parameter-saving cells in order.
3. The final API submission cell is disabled by default. Fill in the live target address and deliberately enable the switch only when you are working on an authorized active instance.

The strategy uses only the provided training labels. It leaves feature rows and the held-out test set unchanged. The local test report is for understanding the behavior; the HTB evaluator remains the final authority.


## 1. Imports, fixed model settings, and challenge controls

The assessment fixes the logistic-regression configuration because the evaluator expects the same model family and parameter layout. POISON_RATE controls what fraction of genuine Class 1 training labels we flip. It is an experimental setting, not a universal best value.

TARGET_HOST and TARGET_PORT are placeholders for a live HTB instance. The assessment page may show a different route or port for a specific instance, so verify those values there before enabling submission.


In [ ]:
from pathlib import Path
import json

import numpy as np
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.multiclass import OneVsRestClassifier

SEED = 1337
CLASS_IDS = np.arange(4)
CLASS_NAMES = [f"Class {class_id}" for class_id in CLASS_IDS]

# Keep these estimator settings exactly as the HTB template specifies.
BASE_ESTIMATOR_CONFIG = {
    "random_state": SEED,
    "solver": "liblinear",
    "C": 1.0,
    "max_iter": 200,
}

DATASET_PATH = Path("assessment_dataset.npz")
OUTPUT_MODEL_PATH = Path("poisoned_model_params.npz")

# Starting experiment setting: flip half of the Class 1 training labels.
# Change this only while exploring training data; do not tune using the test set.
POISON_RATE = 0.50

# The live assessment API is optional. Submission remains disabled by default.
TARGET_HOST = "<HTB_TARGET_HOST>"
TARGET_PORT = "<HTB_TARGET_PORT>"
API_EVALUATOR_URL = f"http://{TARGET_HOST}:{TARGET_PORT}/evaluate_model"
SUBMIT_TO_HTB = False

print("Estimator settings:", BASE_ESTIMATOR_CONFIG)
print("Poison rate:", POISON_RATE)
print("API submission enabled:", SUBMIT_TO_HTB)


## 2. Load and validate the supplied arrays

X contains input features and y contains the associated class labels. A shape of (1500, 2) means 1,500 examples with two measured values each. allow_pickle=False prevents NumPy from loading Python objects from this data file.

We copy the original labels into a clearly named variable so later cells can show exactly which answers changed.


In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATASET_PATH}. Extract it from the HTB assessment ZIP "
        "into the same folder as this notebook."
    )

with np.load(DATASET_PATH, allow_pickle=False) as dataset:
    X_train_orig = dataset["X_train"]
    y_train_orig = dataset["y_train"]
    X_test = dataset["X_test"]
    y_test = dataset["y_test"]

# Fail early if the provided data is incomplete or contains invalid numeric values.
if X_train_orig.ndim != 2 or X_train_orig.shape[1] != 2:
    raise ValueError(f"Expected two training features per row; got {X_train_orig.shape}.")
if X_test.ndim != 2 or X_test.shape[1] != 2:
    raise ValueError(f"Expected two test features per row; got {X_test.shape}.")
if y_train_orig.shape != (len(X_train_orig),):
    raise ValueError("Training feature and label counts do not match.")
if y_test.shape != (len(X_test),):
    raise ValueError("Test feature and label counts do not match.")
if not np.isfinite(X_train_orig).all() or not np.isfinite(X_test).all():
    raise ValueError("Features must be finite numbers.")
if not np.isin(y_train_orig, CLASS_IDS).all() or not np.isin(y_test, CLASS_IDS).all():
    raise ValueError("Labels must be integers from 0 through 3.")

print("X_train:", X_train_orig.shape, X_train_orig.dtype)
print("y_train:", y_train_orig.shape, y_train_orig.dtype)
print("X_test: ", X_test.shape, X_test.dtype)
print("y_test: ", y_test.shape, y_test.dtype)
print("Training count by class:", np.bincount(y_train_orig, minlength=4))
print("Test count by class:    ", np.bincount(y_test, minlength=4))


## 3. Train a clean baseline

For each class k, one-vs-rest (OvR) fits a binary classifier for “class k” versus “all other classes.” With four labels, it fits four classifiers. Each classifier's score is based on a weighted sum of the two features plus a bias.

The clean baseline gives us a reference before any labels are changed. We use the same estimator configuration again for the poisoned model.


In [ ]:
baseline_model = OneVsRestClassifier(
    LogisticRegression(**BASE_ESTIMATOR_CONFIG)
)
baseline_model.fit(X_train_orig, y_train_orig)

baseline_predictions = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print(f"Clean baseline accuracy: {baseline_accuracy:.3f}")
print("Clean baseline confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(y_test, baseline_predictions, labels=CLASS_IDS))
print("Clean baseline class report:")
print(
    classification_report(
        y_test,
        baseline_predictions,
        labels=CLASS_IDS,
        target_names=CLASS_NAMES,
        zero_division=0,
    )
)


## 4. Choose vulnerable Class 1 rows and flip only their labels

For a Class 1 training row x_i, the fitted OvR model supplies scores s_0(x_i), s_1(x_i), s_2(x_i), and s_3(x_i). We compare Class 1 only with the two requested rival labels, 0 and 2.

The margin is m_i = s_1(x_i) - max(s_0(x_i), s_2(x_i)). Read it as “m sub i equals score one at x sub i minus the maximum of score zero and score two at x sub i.” A smaller margin means the baseline gave Class 1 less separation from its strongest requested rival. The selected rows are relabeled toward that stronger rival.

Example: if the three relevant scores are Class 1 1.2, Class 0 1.0, and Class 2 -0.4, then the margin is 1.2 - max(1.0, -0.4) = 0.2. Class 1 wins, but only narrowly. The original feature vector remains untouched; only the training label changes.


In [ ]:
if not 0.0 <= POISON_RATE < 1.0:
    raise ValueError("POISON_RATE must be at least 0 and less than 1.")

# Work on a copy so y_train_orig remains the clean reference.
y_train_poisoned = y_train_orig.copy()
class1_train_indices = np.flatnonzero(y_train_orig == 1)

if len(class1_train_indices) < 2:
    raise ValueError("Need at least two Class 1 training rows to leave some unflipped.")

# decision_function returns one score per class for each row.
class1_scores = baseline_model.decision_function(X_train_orig[class1_train_indices])
requested_rival_scores = class1_scores[:, [0, 2]]

# The stronger of classes 0 and 2 is the destination label for each selected row.
rival_position = np.argmax(requested_rival_scores, axis=1)
rival_label = np.array([0, 2])[rival_position]

# Margin: Class 1 score minus its strongest requested rival score.
class1_margin = class1_scores[:, 1] - requested_rival_scores.max(axis=1)

# Select the lowest-margin examples first; stable sorting makes ties reproducible.
num_to_flip = int(np.floor(POISON_RATE * len(class1_train_indices)))
vulnerable_order = np.argsort(class1_margin, kind="stable")
selected_positions = vulnerable_order[:num_to_flip]
indices_to_flip = class1_train_indices[selected_positions]

# Only selected Class 1 labels change, and each changes to Class 0 or Class 2.
y_train_poisoned[indices_to_flip] = rival_label[selected_positions]

changed_rows = np.flatnonzero(y_train_poisoned != y_train_orig)
assert np.all(y_train_orig[changed_rows] == 1)
assert np.all(np.isin(y_train_poisoned[changed_rows], [0, 2]))

print(f"Class 1 training examples: {len(class1_train_indices)}")
print(f"Labels flipped: {len(changed_rows)}")
print("Original training class counts:", np.bincount(y_train_orig, minlength=4))
print("Poisoned training class counts:", np.bincount(y_train_poisoned, minlength=4))
print("Flips toward Class 0:", int(np.sum(y_train_poisoned[changed_rows] == 0)))
print("Flips toward Class 2:", int(np.sum(y_train_poisoned[changed_rows] == 2)))


## 5. Train and evaluate the poisoned model

We retrain from scratch with the same features, fixed model settings, and altered training labels. The test inputs and answers remain clean. The confusion matrix uses rows for true classes and columns for predicted classes.

The key outcome is not just overall accuracy. The assessment asks where genuine Class 1 test rows go, especially whether both requested destinations, Class 0 and Class 2, receive errors.


In [ ]:
poisoned_model = OneVsRestClassifier(
    LogisticRegression(**BASE_ESTIMATOR_CONFIG)
)
poisoned_model.fit(X_train_orig, y_train_poisoned)

poisoned_predictions = poisoned_model.predict(X_test)
poisoned_accuracy = accuracy_score(y_test, poisoned_predictions)

print(f"Clean baseline accuracy:  {baseline_accuracy:.3f}")
print(f"Poisoned model accuracy: {poisoned_accuracy:.3f}")
print("Poisoned confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(y_test, poisoned_predictions, labels=CLASS_IDS))
print("Poisoned class report:")
print(
    classification_report(
        y_test,
        poisoned_predictions,
        labels=CLASS_IDS,
        target_names=CLASS_NAMES,
        zero_division=0,
    )
)

# Focus on the assessment's true-Class-1 subset.
class1_test_predictions = poisoned_predictions[y_test == 1]
class1_total = len(class1_test_predictions)
if class1_total == 0:
    raise ValueError("The supplied test set contains no true Class 1 examples.")

for destination in CLASS_IDS:
    count = int(np.sum(class1_test_predictions == destination))
    print(
        f"True Class 1 predicted as Class {destination}: "
        f"{count}/{class1_total} ({count / class1_total:.1%})"
    )


## 6. Save the parameter format required by HTB

The evaluator expects one weight array and one bias array for each of the four fitted binary estimators, plus the class order. The explicit names below follow the student template exactly. Saving is local; it does not submit the model.


In [ ]:
if not hasattr(poisoned_model, "estimators_") or len(poisoned_model.estimators_) != 4:
    raise ValueError("Expected four fitted OvR estimators before saving.")

params_to_save = {}
for estimator_index, estimator in enumerate(poisoned_model.estimators_):
    params_to_save[f"coef_estimator_{estimator_index}"] = estimator.coef_
    params_to_save[f"intercept_estimator_{estimator_index}"] = estimator.intercept_

params_to_save["classes_"] = poisoned_model.classes_
np.savez_compressed(OUTPUT_MODEL_PATH, **params_to_save)

print(f"Saved evaluator-format parameters to: {OUTPUT_MODEL_PATH.resolve()}")
print("Stored arrays:", sorted(params_to_save))


## 7. Optional submission to the live HTB evaluator

The endpoint and port vary with the spawned instance. Confirm the active target details in Academy and set TARGET_HOST and TARGET_PORT before enabling submission. The request uses multipart field model_params, matching the official template.

SUBMIT_TO_HTB is False by default. A successful HTTP response is not necessarily an accepted attack; inspect the evaluator's JSON result. Do not use the local test report as a substitute for the evaluator's result.


In [ ]:
if not SUBMIT_TO_HTB:
    print("Submission is disabled. Set SUBMIT_TO_HTB = True only for an active authorized HTB instance.")
else:
    if "<" in TARGET_HOST or ">" in TARGET_HOST or "<" in TARGET_PORT or ">" in TARGET_PORT:
        raise ValueError("Replace the HTB target placeholders before submitting.")
    if not OUTPUT_MODEL_PATH.exists():
        raise FileNotFoundError(f"Cannot submit; {OUTPUT_MODEL_PATH} does not exist.")

    with OUTPUT_MODEL_PATH.open("rb") as model_file:
        response = requests.post(
            API_EVALUATOR_URL,
            files={
                "model_params": (
                    OUTPUT_MODEL_PATH.name,
                    model_file,
                    "application/octet-stream",
                )
            },
            timeout=30,
        )

    print("HTTP status:", response.status_code)
    try:
        evaluator_result = response.json()
        print(json.dumps(evaluator_result, indent=2))
    except requests.exceptions.JSONDecodeError:
        print(response.text)


## COAE quick recall

- OvR means one binary classifier per class: this class versus all remaining classes.
- w_k transpose times x plus b_k is the class score: weighted feature sum plus bias.
- argmax means the index of the largest score.
- Label flipping changes y_i, not the feature vector x_i; keep the test labels clean.
- The assessment specifically watches Class 1 errors into Classes 0 and 2, so inspect those destinations directly.
- Keep the provided estimator settings and parameter names unchanged.
- The API submission switch is off by default and the target host/port remain placeholders.
